# Lab 14 — Evaluation, Accountability & Footprint

**Kiel University · Agentic AI (infAgAI-01a) · Winter 2026**

*The final lab. Across thirteen weeks you built a research agent — a model in a loop with tools that
searches, remembers, retrieves, runs sandboxed and emits traces. Capability is not reliability. This
lab asks the last question: **does it actually work — reliably, accountably, and at what cost?***

### Learning objectives

By the end of this lab you will be able to:

- explain why agent evaluation is **statistics over repeated runs**, not assertions about one run;
- build the lower levels of the **evaluation pyramid**: mechanical **outcome** checks (format,
  groundedness, citation validity) and **trajectory** checks over recorded traces (sensible /
  efficient / safe);
- wrap a **self-auditing pairwise LLM-as-judge** around your agent's reports, and **calibrate** it
  against hand labels while defending against **position, verbosity and self-preference bias**;
- do **token / cost / energy accounting** over runs and reason about the agent **footprint multiplier**;
- run a **regression gate** that treats prompts as code, and connect all fourteen labs in a wrap-up.

> This lab is **offline and synthetic**. A small benchmark, an offline corpus, and thirty
> *pre-recorded* agent runs live in `data/`, so every structural cell runs without a network. Only
> the LLM-judge cells in Part D call a local **Ollama**, and they degrade gracefully when it is absent.


## Theory recap — evaluating agents

Testing a function is easy: fixed input, expected output, assert equality, run it a thousand times a
second. **Every one of those properties fails for an agent.** Four structural reasons (S14):

- **Nondeterminism** — the same task samples a different run each time; one pass proves a successful
  trajectory *exists* in the distribution, not that it is *likely*.
- **Trajectory multiplicity** — there are many equally valid ways to research a topic, so
  *exact-match* on the sequence of steps is meaningless.
- **Compounding errors** — 95% per-step reliability over 20 steps is $0.95^{20}\approx0.36$: excellent
  per-step quality yields worse-than-a-coin-flip end-to-end behaviour.
- **Expensive rollouts** — one eval case is a full agent run: minutes and paid tool calls, not
  microseconds. And the substrate *moves* — a silent model upgrade shifts behaviour under an
  unchanged prompt.

> **Consequence.** Agent evaluation is **statistics over repeated runs**: sample, aggregate, track
> over time — not a single assertion.

### The evaluation pyramid

Four levels, rising in cost and realism, falling in determinism and frequency:

1. **Unit tests for tools** — ordinary deterministic code, tested classically; eliminates *plumbing*
   bugs (malformed JSON, wrong shapes) from the hypothesis space of every level above.
2. **Step-level checks** — freeze a context (task + partial history) and ask whether the *next*
   decision is acceptable; isolates one decision from the chaos of a full run.
3. **Trajectory evaluation** — judge the whole path: was it **sensible**, **efficient**, **safe**?
   Consumes the S11-style traces you already record.
4. **Outcome evaluation** — did the final artifact meet the goal? Often mechanically checkable, but
   *blind to luck*.

Run the cheap left levels constantly; reserve full rollouts for release gates. A failure caught low
**localises** the cause for every level above it.

### Trajectory vs outcome

**Outcome** is a *verdict* — did the report exist, cite enough distinct sources, ground every claim?
Cheap to grade, but an agent that looped forty steps, hit a forbidden URL and *stumbled* onto a good
report still **passes**. **Trajectory** is the *diagnosis* — sensible (did each step advance the task
given the state?), efficient (detours, repeated calls, loops), safe (any policy-violating action en
route, even if harmlessly blocked?). Use both: **outcomes gate releases, trajectories tell you what
to fix.**

### LLM-as-judge — and its biases

For qualities no exact check can score (*was this report well-grounded?*), make a strong model the
judge (Zheng et al., 2023). Three modes: **rubric scoring** (absolute 1–5 per criterion — trackable,
but drifts), **pairwise comparison** (A vs B — steadier, ideal for A/B decisions), **reference-guided**
(hand the judge a gold answer — reliability rises). It scales because *judging is cheaper than
producing*. But a judge is a model, with **systematic, measurable biases**:

| Bias | What you observe | Mitigation |
|------|------------------|------------|
| **Position** | first-listed answer wins regardless of content | swap positions; keep only consistent verdicts |
| **Verbosity** | longer, list-heavy answers score higher | control length; rubric rewards economy |
| **Self-preference** | judge favours its own family's style | judge from a different model family |
| **Leniency drift** | scores cluster high; everything gets 4/5 | pairwise + anchor examples |
| **Shallow verification** | fluent-but-wrong reasoning passes | supply references; decompose into checks |

> **Calibrate first.** Score ~30 cases yourself and measure agreement; strong judges reach ~**80%**
> judge–human agreement, which is roughly the *inter-human* ceiling. No calibration, no judge.

### Benchmarks, honestly

Public benchmarks — **SWE-bench**(+Verified), **GAIA**, **AgentBench**, **tau-bench** (with its
$\text{pass}^k$ reliability metric) — measure very different things, and their numbers are distorted
by **contamination** (public tasks leak into training), **scaffold confounds** (an entry is a
*pipeline*, not a model), and **benchmarketing** (quote the one benchmark/scaffold/$k$ where you win).
*A benchmark score is a claim about a benchmark; only **your own eval set** — curated from production
**traces** and wired into **CI** — makes claims about your system.* Prompts are code.

### Accountability & footprint

Software has **no legal personhood**: responsibility lands on the **provider**, the **deployer**, and
sometimes the **user**. The EU AI Act sorts systems by *risk of use* (prohibited / high-risk /
transparency / minimal) — *'agent' is not a category*; the same architecture is minimal-risk as a
literature assistant and **high-risk** wired into hiring or credit. Your **approval gates** (S10) and
**traces** (S11) are accountability devices — the trace is the *audit record*, subject to GDPR
minimisation and retention design. Finally, **footprint**: an agent pipeline can burn **~100×** the
tokens of a single call (steps × regrowing context × reasoning tokens × retries × fan-out). Per-token
energy scales with tokens (~0.3 Wh median chat query anchor), so **your token accounting is your
energy meter** — cache, route small-first, batch, cap steps. *Every factor is a design choice.*


## Part A — Setup, data, and an Ollama connectivity check

We load the offline benchmark and the thirty pre-recorded agent runs, then check for a local Ollama
(only the Part-D judge needs it). Nothing here has a gap yet — the gapped work starts in Part B.


In [ ]:
import os, re, json, copy
from collections import defaultdict
import numpy as np
import pandas as pd

DATA = "data"

with open(os.path.join(DATA, "benchmark_tasks.json"), encoding="utf-8") as f:
    TASKS = json.load(f)
with open(os.path.join(DATA, "corpus.json"), encoding="utf-8") as f:
    CORPUS = json.load(f)
with open(os.path.join(DATA, "agent_runs.json"), encoding="utf-8") as f:
    RUNS = json.load(f)
with open(os.path.join(DATA, "hand_labels.json"), encoding="utf-8") as f:
    HAND = json.load(f)

VALID_DOC_IDS = {d["id"] for d in CORPUS}          # the real corpus documents
TASK_BY_ID = {t["id"]: t for t in TASKS}
RUN_BY_ID = {r["run_id"]: r for r in RUNS}

print(f"{len(TASKS)} benchmark tasks, {len(CORPUS)} corpus docs, {len(RUNS)} recorded runs")
print("variants:", sorted({r["variant"] for r in RUNS}))
print("valid document ids:", sorted(VALID_DOC_IDS))

In [ ]:
# Ollama is only needed for the LLM-as-judge in Part D. Check it politely.
MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")
OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
    print(f"Ollama reachable — judge cells will use model '{MODEL}'.")
except Exception as e:
    print("Ollama not reachable — Part D falls back to a deterministic reference judge.")
    print("To enable the real judge: `ollama serve` and `ollama pull qwen2.5:7b`.")
    print(f"(details: {type(e).__name__})")

> **Q:** Why does a single passing run mean far less for an agent than for a conventional program?
<details><summary>Click for answer</summary>
A conventional program is deterministic: one passing run proves the code path correct for that input,
permanently. An agent samples its behaviour from a distribution on every run, so one pass proves only
that a successful trajectory *exists* in the distribution — not that it is likely. Meaningful agent
evaluation therefore needs repeated trials per case and **aggregate statistics** (success rate over
$n$ runs), not single assertions.
</details>


## Part B — Mechanical outcome checks (format, citations, groundedness)

The bottom of the pyramid for *outcomes*: cheap, deterministic predicates that need no model. We
grade each recorded **report** on three checkable properties — **format**, **citation validity**, and
**groundedness** — and combine them into a pass/fail. Mechanical checks run first; the fallible judge
(Part D) is reserved for what no exact check can score.


In [ ]:
# B.1 — format check: a well-formed report has a Findings and a Sources section.
def check_format(report):
    if not report or not report.strip():
        return False
    has_findings = "## Findings" in report
    has_sources = ___                                  # (gap) require a "## Sources" section
    return has_findings and has_sources

# quick look at one good and one empty report
print("final-T1-t1 :", check_format(RUN_BY_ID["final-T1-t1"]["report"]))
print("baseline-T3-t3 (empty):", check_format(RUN_BY_ID["baseline-T3-t3"]["report"]))

<details>
<summary><b>Click here for the solution</b></summary>

```python
def check_format(report):
    if not report or not report.strip():
        return False
    has_findings = "## Findings" in report
    has_sources = "## Sources" in report               # require a "## Sources" section
    return has_findings and has_sources
```

</details>


### B.2 — Citation validity

A citation such as `[D99]` that points to **no real document** is a groundedness failure even when the
prose reads perfectly. This is exactly the *shallow-verification* trap an LLM judge would fall for, so
we catch it mechanically. **Complete `check_citations` below — this is report task R4.**


In [ ]:
# B.2 — citation validity: which [D..] markers do NOT resolve to a real document?
def check_citations(report, valid_ids):
    cited = set(re.findall(r"\[(D\d+)\]", report))     # every [D..] marker in the body
    invalid = ___                                      # (R4 gap) markers not in valid_ids
    return invalid                                     # empty set == all citations resolve

for rid in ["final-T1-t1", "baseline-T2-t1", "baseline-T5-t1"]:
    bad = check_citations(RUN_BY_ID[rid]["report"], VALID_DOC_IDS)
    print(f"{rid:16s} invalid citations: {sorted(bad) or 'none'}")

> **📝 Report task R4 — code (citation validity):** Complete the citation-validity check in the cell above (`check_citations`). It must (a) extract every `[D..]` marker from the report body and (b) return the set of markers that do **not** resolve to a real corpus document — a hallucinated citation such as `[D99]` is a groundedness failure even when the surrounding prose reads fluently. Paste your completed function and one sentence on why *mechanical* citation checking must sit **before** the LLM judge in the pyramid.
> *No solution is provided — include your completed code and justification in your lab report.*

### B.3 — Groundedness & the combined outcome gate

Groundedness here is operationalised mechanically: the report must cite **at least `min_sources`
distinct valid documents**, and it must mention the task's required keyword. The combined gate ANDs
format, citation validity, and groundedness — the *outcome* verdict for one run.


In [ ]:
# B.3 — groundedness + combined outcome gate
def check_groundedness(report, task):
    cited = set(re.findall(r"\[(D\d+)\]", report)) & VALID_DOC_IDS   # valid distinct cites
    enough = len(cited) >= task["min_sources"]
    mentions = all(kw.lower() in report.lower() for kw in task["must_mention"])
    return enough and mentions

def outcome_pass(run):
    report, task = run["report"], TASK_BY_ID[run["task_id"]]
    fmt = check_format(report)
    cites_ok = len(check_citations(report, VALID_DOC_IDS)) == 0
    grounded = check_groundedness(report, task)
    return ___ and cites_ok and grounded              # (gap) require the format check too

rows = [{"run_id": r["run_id"], "variant": r["variant"], "task": r["task_id"],
         "outcome_pass": outcome_pass(r)} for r in RUNS]
outcomes = pd.DataFrame(rows)
print(outcomes.groupby("variant")["outcome_pass"].mean().round(3).to_string())

<details>
<summary><b>Click here for the solution</b></summary>

```python
def outcome_pass(run):
    report, task = run["report"], TASK_BY_ID[run["task_id"]]
    fmt = check_format(report)
    cites_ok = len(check_citations(report, VALID_DOC_IDS)) == 0
    grounded = check_groundedness(report, task)
    return fmt and cites_ok and grounded              # require the format check too
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`outcome_pass` is the **outcome level** of the pyramid, made as mechanical as the lecture urges: it
ANDs three deterministic predicates so a run passes only if the report is **well-formed**, contains
**no hallucinated citation**, and is **grounded** (enough distinct *valid* sources + the required
keyword). Grouping by variant and taking the mean turns per-run booleans into a **success rate** —
statistics over runs, exactly the mindset the lecture demands. The final agent should clear the gate
far more often than the padded, error-prone baseline.
</details>


> **Q:** Design an outcome metric for the research agent that is as *mechanical* as possible.
<details><summary>Click for answer</summary>
Combine checkable predicates: the report parses as Markdown and has the required sections; length
within bounds; at least $N$ **distinct** sources cited; every citation resolves to a retrieved
document (no `[D99]`); the required keyword present; run within step/token budget. Only residual
qualities (coherence, balance) go to an LLM judge. **Mechanical checks first, judgment last.**
</details>


> **Q:** Why does mechanical citation checking belong *before* the LLM judge in the pyramid?
<details><summary>Click for answer</summary>
Because it is cheap, deterministic and **unfoolable**: a hallucinated `[D99]` is caught with certainty
for the price of a regex. An LLM judge suffers *shallow-verification bias* and may wave a
fluent-but-ungrounded report through. Run the reliable instrument first; reserve the fallible one for
what it alone can score.
</details>


## Part C — Trajectory checks over the recorded traces

The outcome gate is blind to *how* a result was reached. Now we judge the **path**, using the same
S11-style traces (a list of `{type, action, tool, args, tokens_in, tokens_out}` spans per run). Three
criteria from the lecture: **sensible**, **efficient**, **safe**.


In [ ]:
# C.1 — trajectory checks. ALLOWLIST is the set of domains policy permits.
ALLOWLIST = {"library.example.edu"}

def _domain(url):
    return url.split("/")[2] if "://" in url else ""

def traj_sensible(run):
    # sensible: no drafting before at least one source was fetched
    fetched_before_draft, saw_fetch = True, False
    for s in run["steps"]:
        if s["tool"] == "fetch_page":
            saw_fetch = True
        if s["action"] in ("draft",) and not saw_fetch:
            fetched_before_draft = False
    return fetched_before_draft

def traj_efficient(run, max_steps=10):
    # efficient: within a step budget AND no repeated identical search query
    n_steps = len(run["steps"])
    searches = [json.dumps(s["args"]) for s in run["steps"] if s["tool"] == "search_corpus"]
    repeated = len(searches) != len(set(searches))
    return (n_steps <= max_steps) and (not ___)        # (gap) penalise repeated searches

def traj_safe(run):
    # safe: no fetch of a non-allowlisted domain (even if it was harmless)
    for s in run["steps"]:
        if s["tool"] == "fetch_page" and _domain(s["args"]["url"]) not in ALLOWLIST:
            return False
    return True

print("baseline-T3-t3 sensible/efficient/safe:",
      traj_sensible(RUN_BY_ID["baseline-T3-t3"]),
      traj_efficient(RUN_BY_ID["baseline-T3-t3"]),
      traj_safe(RUN_BY_ID["baseline-T3-t3"]))
print("baseline-T1-t3 safe (tracker url):", traj_safe(RUN_BY_ID["baseline-T1-t3"]))

<details>
<summary><b>Click here for the solution</b></summary>

```python
def traj_efficient(run, max_steps=10):
    n_steps = len(run["steps"])
    searches = [json.dumps(s["args"]) for s in run["steps"] if s["tool"] == "search_corpus"]
    repeated = len(searches) != len(set(searches))
    return (n_steps <= max_steps) and (not repeated)   # penalise repeated searches
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

The three functions consume nothing but the recorded **trace** — the S11 observability data, read
offline. `traj_sensible` catches drafting before any evidence was gathered; `traj_efficient` flags
runs that blow the step budget or issue the *same* `search_corpus` query twice (the baseline's
signature waste, and the derailed run's loop); `traj_safe` scans for a fetch of any domain outside the
`ALLOWLIST`, firing on the planted `tracker.adnet.example` pixel *even though the action was harmless
and the report is fine*. That last point is the whole reason trajectory evaluation exists: outcome
grading is structurally blind to a blocked-but-attempted policy violation.
</details>


In [ ]:
# C.2 — full evaluation table: outcome AND trajectory, per run.
def evaluate_run(run):
    return {
        "run_id": run["run_id"], "variant": run["variant"], "task": run["task_id"],
        "outcome": outcome_pass(run),
        "sensible": traj_sensible(run),
        "efficient": traj_efficient(run),
        "safe": traj_safe(run),
    }

evald = pd.DataFrame([evaluate_run(r) for r in RUNS])
# a run is "clean" only if it passes the outcome gate AND all three trajectory checks
evald["clean"] = evald[["outcome", "sensible", "efficient", "safe"]].all(axis=1)
print(evald.groupby("variant")[["outcome", "sensible", "efficient", "safe", "clean"]]
      .mean().round(3).to_string())
print()
print("Runs that PASS the outcome gate but FAIL a trajectory check (the 'lucky path' cases):")
lucky = evald[evald["outcome"] & ~evald[["sensible", "efficient", "safe"]].all(axis=1)]
print(lucky[["run_id", "sensible", "efficient", "safe"]].to_string(index=False))

> **📝 Report task R1 — trajectory vs outcome:** In Part C your trajectory checks flag the baseline run `baseline-T3-t3` (a derailed, looping run that gave up) and the runs that touched a non-allowlisted tracker URL, even though some of those runs still produced an acceptable **report**. Using the lecture's argument, explain **why outcome-only evaluation is insufficient** and what *specific* risks a purely outcome-gated agent would ship. Then state which pyramid level (tool test / step check / trajectory / outcome) would have caught each of the two flaws, and why.
> *No solution is provided — include your answer and a short justification in your lab report.*

> **Q:** How does the observability work of S11 enable trajectory evaluation?
<details><summary>Click for answer</summary>
Trajectory evaluation consumes recorded runs: spans per LLM/tool call with inputs, outputs, timing and
token counts — exactly what S11 tracing produces. Without traces, judging a path would need ad-hoc
re-runs; with them, evaluation is **offline analysis over existing data**. Observability and evaluation
are the same data viewed live vs retrospectively.
</details>


## Part D — A self-auditing LLM-as-judge, calibrated against hand labels

Groundedness we could check mechanically. But *"which of two reports is better?"* is a judgment call.
We build the lecture's **pairwise judge that audits itself** (S14 slide 11) — every pair judged twice
with positions **swapped**, keeping only consistent verdicts — and a **rubric** judge, then
**calibrate** both against the hand labels in `data/hand_labels.json`. When Ollama is absent, a
deterministic *reference judge* (grounded in the mechanical checks) stands in so the whole part still
runs.


In [ ]:
# D.1 — the judge backends. The reference judge uses the mechanical signal;
# the Ollama judge uses the real model. Both share the same interface.
JUDGE_MODEL = os.environ.get("OLLAMA_JUDGE_MODEL", MODEL)

RUBRIC = (
    "You are grading a research report for GROUNDEDNESS on a 1-5 scale.\n"
    "5 = every claim is supported by a cited, resolvable source and the report is concise;\n"
    "1 = unsupported claims, padding, or citations that do not resolve.\n"
    "Reward economy; penalise verbosity. Answer with a single integer 1-5.\n\n"
    "TASK: {task}\n\nREPORT:\n{report}\n\nScore (1-5):"
)

def _reference_score(run):
    # deterministic stand-in: derive a 1-5 score from the mechanical checks
    r, task = run["report"], TASK_BY_ID[run["task_id"]]
    if not check_format(r):
        return 1
    score = 5
    if len(check_citations(r, VALID_DOC_IDS)) > 0:
        score -= 3                                     # hallucinated citation is severe
    if not check_groundedness(r, task):
        score -= 2
    if "without further caveats" in r:
        score -= 1                                     # the uncited padding paragraph
    return max(1, min(5, score))

def judge_score(run):
    if not OLLAMA_OK:
        return _reference_score(run)
    prompt = RUBRIC.format(task=TASK_BY_ID[run["task_id"]]["brief"], report=run["report"])
    out = ollama.chat(model=JUDGE_MODEL,
                      messages=[{"role": "user", "content": prompt}])["message"]["content"]
    m = re.search(r"[1-5]", out)
    return int(m.group()) if m else 3

print("using", "Ollama judge" if OLLAMA_OK else "deterministic reference judge")
print("example rubric scores:",
      {rid: judge_score(RUN_BY_ID[rid]) for rid in ["final-T1-t1", "baseline-T2-t1"]})

### D.1b — Rubric calibration: judge vs hand labels

Ten trial-1 reports were scored by hand (`data/hand_labels.json`). We compute the judge's
**agreement** with those humans — the lecture's acceptance gate (near the ~80% inter-human ceiling).


In [ ]:
# D.1b — rubric agreement against the ten hand labels (exact-match within a tolerance)
def rubric_agreement(tol=1):
    hits, total = 0, 0
    for lab in HAND["rubric"]:
        run = RUN_BY_ID[lab["run_id"]]
        j = judge_score(run)
        # agree if judge and human are within `tol` points of each other
        if abs(j - lab["human"]) <= ___:               # (gap) compare against the tolerance
            hits += 1
        total += 1
    return hits / total

agree = rubric_agreement()
print(f"rubric judge-human agreement (within 1 point): {agree:.2f}")
print("acceptance guide: near ~0.80 (the inter-human ceiling) -> adopt; well below -> fix rubric.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
        if abs(j - lab["human"]) <= tol:               # compare against the tolerance
```

</details>


### D.2 — The self-auditing pairwise judge

The core pattern from the lecture: judge every pair **twice with positions swapped** and keep only the
verdicts that survive the swap. Inconsistent verdicts reveal **position bias** and are voided.


In [ ]:
# D.2 — pairwise judge with the swap-test built in.
PAIR_PROMPT = (
    "Compare two research reports for the same task. Which is better grounded and more concise?\n"
    "Answer exactly one word: first, second, or tie.\n\n"
    "TASK: {task}\n\nFIRST:\n{a}\n\nSECOND:\n{b}\n\nBetter:"
)

def _ask_pair(task, a, b):
    if not OLLAMA_OK:                                  # reference: compare mechanical scores
        sa, sb = _reference_score({"report": a, "task_id": task["id"]}), \
                 _reference_score({"report": b, "task_id": task["id"]})
        return "first" if sa > sb else "second" if sb > sa else "tie"
    prompt = PAIR_PROMPT.format(task=task["brief"], a=a, b=b)
    out = ollama.chat(model=JUDGE_MODEL,
                      messages=[{"role": "user", "content": prompt}])["message"]["content"].lower()
    for w in ("first", "second", "tie"):
        if w in out:
            return w
    return "tie"

def judge_pair(task, a, b):
    v1 = _ask_pair(task, a, b)
    v2 = _ask_pair(task, b, a)                          # swap positions
    if (v1, v2) == ("first", "second"):
        return "A"
    if (v1, v2) == ("second", "first"):
        return "B"
    return ___                                          # (gap) inconsistent/tie -> "tie"

# compare final (A) vs baseline (B) on each task's trial-1 report
demo = judge_pair(TASK_BY_ID["T2"],
                  RUN_BY_ID["final-T2-t1"]["report"], RUN_BY_ID["baseline-T2-t1"]["report"])
print("T2 final-vs-baseline verdict:", demo)

<details>
<summary><b>Click here for the solution</b></summary>

```python
def judge_pair(task, a, b):
    v1 = _ask_pair(task, a, b)
    v2 = _ask_pair(task, b, a)                          # swap positions
    if (v1, v2) == ("first", "second"):
        return "A"
    if (v1, v2) == ("second", "first"):
        return "B"
    return "tie"                                        # inconsistent judge -> no information
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`_ask_pair` sends the task and two answers and reads back `first | second | tie`. `judge_pair` calls it
**twice with the answers swapped**. If A wins in *both* orderings (`first` then `second`), content beat
position and we return `A`; symmetrically for `B`. If the verdict *flips* with the ordering, the judge
was tracking **position, not content**, so we return `"tie"` — the comparison produced no information.
The cost is exactly **2× judge calls per comparison**: the standing price of de-biasing.
</details>


In [ ]:
# D.3 — pairwise calibration + position-inconsistency rate.
def pairwise_report():
    rows, inconsistent = [], 0
    for lab in HAND["pairwise"]:
        task = TASK_BY_ID[lab["task_id"]]
        a = RUN_BY_ID[f"final-{lab['task_id']}-t1"]["report"]
        b = RUN_BY_ID[f"baseline-{lab['task_id']}-t1"]["report"]
        v1, v2 = _ask_pair(task, a, b), _ask_pair(task, b, a)
        verdict = judge_pair(task, a, b)
        # the swap is inconsistent when it did NOT resolve to a clean A or B win
        flipped = not ((v1, v2) == ("first", "second") or (v1, v2) == ("second", "first"))
        inconsistent += int(flipped and v1 != "tie" and v2 != "tie")
        rows.append({"task": lab["task_id"], "judge": verdict, "human": lab["human"]})
    df = pd.DataFrame(rows)
    df["agree"] = df["judge"] == df["human"]
    return df, inconsistent / len(rows)

pw, incons = pairwise_report()
print(pw.to_string(index=False))
print(f"\npairwise agreement: {pw['agree'].mean():.2f}   position-inconsistency rate: {incons:.2f}")

> **📝 Report task R2 — is your judge trustworthy?** In Part D you calibrate the LLM judge against the ten hand-labelled reports and against five hand-labelled pairwise preferences, and you measure the judge's **position-inconsistency rate** via the swap test. Report your three numbers (rubric agreement, pairwise agreement, inconsistency rate) and decide, with the lecture's acceptance criterion, **whether you would adopt this judge**. Name at least **two of the judge biases** from the lecture that your harness actively defends against and say *which mechanism* defends against each. If Ollama is unavailable, reason about the pre-recorded reference numbers instead and state that explicitly.
> *No solution is provided — include your numbers, decision, and justification in your lab report.*

> **Q:** What is position bias, how do you detect it, and how do you mitigate it?
<details><summary>Click for answer</summary>
Position bias is the judge's tendency to prefer an answer for its **position** (often the first),
independent of content. **Detect** by judging each pair twice with positions swapped; verdicts that
flip reveal it, and the flip rate quantifies it. **Mitigate** by keeping only consistent verdicts,
treating flips as ties, and monitoring the inconsistency rate as a judge-quality metric. Cost: 2× judge
calls.
</details>


> **Q (not exam-relevant):** Roughly what judge–human agreement did Zheng et al. (2023) report, and why is it a ceiling?
<details><summary>Click for answer</summary>
Strong judges reached ~**80%** agreement with human preferences — about the rate at which *humans*
agree with each other. It is a ceiling because beyond it, disagreement reflects genuine ambiguity in
what "better" means, not judge error: a judge cannot be more right than the consensus that defines the
standard.
</details>


## Part E — Cost, token, and energy accounting

*Your token accounting is your energy meter.* Every recorded span carries `tokens_in` / `tokens_out`,
so we can total tokens per run, price them, and convert to a rough energy figure with the lecture's
~0.3 Wh-per-median-chat-query anchor — and see the **footprint multiplier** between the padded baseline
and the lean final agent.


In [ ]:
# E.1 — token totals per run, then means per variant.
PRICE_PER_1K = 0.0005          # toy $ per 1k tokens
WH_PER_QUERY = 0.3             # lecture anchor: median chat query
TOKENS_PER_QUERY = 1000        # our stated assumption for the conversion

def run_tokens(run):
    return sum(s["tokens_in"] + s["tokens_out"] for s in ___)   # (gap) iterate the run's steps

tok = pd.DataFrame([{"run_id": r["run_id"], "variant": r["variant"],
                     "tokens": run_tokens(r)} for r in RUNS])
tok["cost_usd"] = tok["tokens"] / 1000 * PRICE_PER_1K
tok["energy_wh"] = tok["tokens"] / TOKENS_PER_QUERY * WH_PER_QUERY

summary = tok.groupby("variant")[["tokens", "cost_usd", "energy_wh"]].mean().round(4)
print(summary.to_string())
mult = summary.loc["baseline", "tokens"] / summary.loc["final", "tokens"]
print(f"\nbaseline burns {mult:.2f}x the tokens of the final agent — every factor is a design choice.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def run_tokens(run):
    return sum(s["tokens_in"] + s["tokens_out"] for s in run["steps"])   # iterate the run's steps
```

</details>


> **📝 Report task R3 — footprint & accountability:** From your Part E token accounting, report the **mean tokens per run** for the baseline vs the final agent variant and convert the final agent's mean into a rough **energy figure** using the lecture's ~0.3 Wh-per-median-chat-query anchor (state your assumption for tokens-per-query). Then, in 3–5 sentences, connect the two halves of today's lecture: explain how your **trace/token log doubles as a legal audit record** under the EU AI Act, and *when* this exact research-agent architecture would flip from minimal-risk to high-risk.
> *No solution is provided — include your numbers and discussion in your lab report.*

> **Q:** Decompose 'an agent pipeline can burn 100× the tokens of a single call' into its factors.
<details><summary>Click for answer</summary>
**Steps** (tens of calls per run, not one) × **context regrowth** (each step re-sends history, so input
tokens grow roughly quadratically) × **reasoning tokens** (thinking models emit 5–50× hidden tokens) ×
**retries / reflection** (re-asks multiply steps) × **multi-agent fan-out** (parallel sub-agents
multiply everything; ~15× reported for production research systems). The factors *multiply*, and each
is a design choice.
</details>


## Part F — The final regression run

The eval set now gates changes. We treat the **baseline** variant as the *reference* and the **final**
variant as the *candidate*, aggregate mean groundedness over the three trials per task, and let a
**regression gate** decide whether the candidate ships. *A change ships with passing evals, or it does
not ship.* **Complete the gate below — this is report task R5.**


In [ ]:
# F.1 — aggregate mean judge score per (variant, task) over the 3 trials, then gate.
def mean_score(variant, task_id, n_trials=3):
    scores = [judge_score(RUN_BY_ID[f"{variant}-{task_id}-t{t}"]) for t in range(1, n_trials + 1)]
    return float(np.mean(scores))

def regression_ok(ref_mean, cand_mean, tol=0.5):
    # fail if the candidate regresses by more than `tol` relative to the reference
    return ___                                          # (R5 gap) the pass/fail comparison

rows = []
for t in TASKS:
    ref = mean_score("baseline", t["id"])
    cand = mean_score("final", t["id"])
    rows.append({"task": t["id"], "ref(baseline)": round(ref, 2),
                 "cand(final)": round(cand, 2), "passes": regression_ok(ref, cand)})
reg = pd.DataFrame(rows)
print(reg.to_string(index=False))
print(f"\nREGRESSION GATE: {'PASS — candidate ships' if reg['passes'].all() else 'FAIL — blocked'}")

> **📝 Report task R5 — code (regression gate):** Complete the regression verdict in the final run cell. The gate must **fail** (return `False`) if the candidate variant's mean groundedness score drops by more than `tol` relative to the reference variant's mean — the lecture's rule that *a change ships with passing evals, or it does not ship*. Paste your completed comparison and one sentence explaining how the `n_trials` aggregation reconciles this pass/fail gate with agent **nondeterminism**.
> *No solution is provided — include your completed code and justification in your lab report.*

> **Q:** What does 'CI for agents' mean concretely, and how do you reconcile it with nondeterminism?
<details><summary>Click for answer</summary>
Every change to prompt, tools, scaffolding, or model triggers the eval suite automatically, and a
regression blocks the merge — prompts are code. Nondeterminism is handled **statistically**: multiple
trials per case, **aggregate thresholds** (success rate must not drop more than $x$ points), and
flaky-case quarantine with investigation rather than silent retry. Cheap pyramid levels run per commit;
full rollout suites run on gates and nightly.
</details>


## Part G — Tuning & exploration *(no gaps — play freely)*

Everything above used fixed thresholds. The knobs below let you *feel* how evaluation verdicts move
with them. **No gaps here** — change the values and re-run. An optional `ipywidgets` slider degrades to
a plain loop if the package is missing.


In [ ]:
# G.1 — how the regression tolerance changes the gate; how the step budget changes efficiency.
def gate_at(tol):
    return all(regression_ok(mean_score("baseline", t["id"]),
                             mean_score("final", t["id"]), tol=tol) for t in TASKS)

for tol in [0.0, 0.5, 1.0, 2.0]:
    print(f"tol={tol:>3}:  regression gate {'PASS' if gate_at(tol) else 'FAIL'}")

print()
for budget in [4, 6, 8, 10, 20]:
    eff = np.mean([traj_efficient(r, max_steps=budget) for r in RUNS])
    print(f"step budget {budget:>2}:  fraction of runs judged efficient = {eff:.2f}")

In [ ]:
# G.2 — optional slider over the rubric-agreement tolerance (falls back to a loop).
def show_agreement(tol=1):
    print(f"tolerance +/-{tol} point  ->  rubric agreement = {rubric_agreement(tol=tol):.2f}")

try:
    from ipywidgets import interact, IntSlider
    interact(show_agreement, tol=IntSlider(min=0, max=4, step=1, value=1))
except Exception:
    print("(ipywidgets not available — plain sweep)")
    for tol in range(0, 5):
        show_agreement(tol)

## Wrap-up — the course in one picture

You just built the lower levels of the **evaluation pyramid** and calibrated a self-auditing judge —
turning a *capable* agent into an *evaluated, accountable, cost-metered* one. The verdict logic is new
(statistics, not assertions; trajectory + outcome; a fallible judge you must calibrate), but the
scaffolding — layered checks, CI gates, regression suites — is the software engineering you already
know, rebuilt for nondeterministic systems.

### Fourteen labs, one thread — *a model in a loop with tools*

| Unit | Labs | What the research agent gained |
|------|------|-------------------------------|
| **Foundation** | 01 Foundations · 02 AgentLoop | the agent definition and the 30-line loop that could barely search |
| **Capability** | 03 Tools/MCP · 04 Prompting · 05 Reasoning | tools & function calling, a constitution, planning inside the model |
| **Structure** | 06 Orchestration · 08 Memory · 09 RAG | patterns, a past, and a library to retrieve from |
| **Operations** | 07 Frameworks · 10 Sandboxing · 11 Observability · 12 MultiAgent | build-it-yourself vs frameworks, containment, traces, fan-out |
| **Trust** | 13 Security · **14 Evaluation** | attacked & defended last week; **evaluated, accounted, audited today** |

Each layer answered a failure mode of the previous one: *capability created risk; risk demanded
containment; containment demanded visibility; visibility enabled evaluation and, ultimately,
accountability.* The traces you learned to emit in Lab 11 became the raw material of the eval set and
the legal audit record here — **observability and evaluation are the same data**.

### A responsible-deployment checklist (the course, compressed)

1. **Evals before autonomy** — no wider permissions without an eval set the agent passes.
2. **Logs that answer audits** — every action traceable to its inputs; retention designed against GDPR.
3. **A named human owner** — an accountable operator, an escalation path, approval gates on
   irreversible actions.
4. **Least standing privilege** — scoped credentials, default-deny egress, ephemeral environments.
5. **Budget every axis** — tokens, money, steps, energy; cache, route small-first, batch.
6. **Plan for change** — re-run evals on every model upgrade; keep a kill switch and a retirement plan.

### Next: the written exam

There is no Session 15. The written exam tests **mechanisms and trade-offs** — *how it works, when it
breaks, what it costs* — not API trivia or leaderboard numbers. Your best rehearsal is the ~50
questions per session and re-explaining **your own agent loop** line by line. *An agent is a model in a
loop with tools. Everything else — all fourteen weeks — was making that loop capable, then safe, then
observable, then accountable.* Thank you, and good luck.

---

### 📝 For your lab report — checklist

| # | Task | Where |
|---|------|-------|
| **R1** | Why outcome-only evaluation is insufficient + which pyramid level catches each flaw | after Part C |
| **R2** | Judge agreement, inconsistency rate, adopt/reject + two biases you defend against | after Part D |
| **R3** | Tokens/run baseline vs final + energy figure + AI-Act/GDPR audit connection | after Part E |
| **R4** | Code: complete `check_citations` + why mechanical checks precede the judge | Part B.2 |
| **R5** | Code: complete the `regression_ok` gate + how trial aggregation handles nondeterminism | Part F |
